<strong>How to use</strong>
- Don't click 'Run All' because there is more than one strategy of data splitting.
- Instead, click code by code from importing up until executing.
- Click only one desired strategy.

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.model_selection import GroupKFold

## Importing CSV file

In [ ]:
# load dataset and parse datetime
df = pd.read_csv('measurements.csv')
df.info()
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
df_loc = pd.read_csv('../locations.csv')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2119310 entries, 0 to 2119309
Data columns (total 17 columns):
 #   Column         Dtype  
---  ------         -----  
 0   id             int64  
 1   location_key   object 
 2   timestamp_utc  object 
 3   pm25           float64
 4   temperature_c  float64
 5   humidity_pct   float64
 6   pm1            float64
 7   pm10           float64
 8   co2            float64
 9   tvoc           float64
 10  has_pm1        object 
 11  has_pm10       object 
 12  has_co2        object 
 13  has_tvoc       object 
 14  source         object 
 15  raw_payload    float64
 16  inserted_at    object 
dtypes: float64(8), int64(1), object(8)
memory usage: 274.9+ MB


In [4]:
def display_sizes(train, val, test):
    print(f"Train: {train.shape[0]} rows")
    print(f"  Val: {val.shape[0]} rows")
    print(f" Test: {test.shape[0]} rows")

### <strong> [Strategy 1] Chronological Split </strong>
- Data are split based on time-interval using column name `timestamp_utc`.   
- Best for validating air quality forecasting (predicting future trends)

<strong>Data split</strong>
- Train Set (~70%): 70% of earliest entries based on `timestamp_utc`
- Validation Set (~15%): between train and test set
- Test Set (~15%): 15% of most recent entries based on `timestamp_utc`

In [ ]:
# Establish clear chronological boundaries
df = df.sort_values('timestamp_utc').reset_index(drop=True)
n = len(df)
p70 = int(n * 0.70)
p85 = int(n * 0.85) 

train_df = df.iloc[0:p70, :]
val_df = df.iloc[p70:p85, :]
test_df = df.iloc[p85:n, :]

display_sizes(train_df, val_df, test_df)

Train Shape: 116458 rows
Val Shape: 24956 rows
Test Shape: 24956 rows


### <strong> [Strategy 2] LOSO-CV Split </strong>
- Leave-One-Station-Out (LOSO)
- Best use for Spatial Downscaling or Land Use Regression (LUR)
- Splits rows based on `location_key` values. There are 29 unique values in that column.


<strong>Data split</strong>
- Train set (~80%): entries from any of 23 stations 
- Val set (~10%): entries from any of 3 stations 
- Test set (~10%): entries from any of 3 stations 

In [26]:
unique_locations = df['location_key'].unique()
print(f"toal available spatial nodes: {len(unique_locations)}")

np.random.seed(42)
shuffled_locations = np.random.permutation(unique_locations)

test_locs = shuffled_locations[:3]
val_locs = shuffled_locations[3:6]
train_locs = shuffled_locations[6:]

train_df = df[df['location_key'].isin(train_locs)]
val_df = df[df['location_key'].isin(val_locs)]
test_df = df[df['location_key'].isin(test_locs)]

display_sizes(train_df, val_df, test_df)

toal available spatial nodes: 29
Train: 150088 rows
  Val: 6924 rows
 Test: 9358 rows


### <strong> [Strategy 3] Andrei's Suggestion </strong>
- Most non-bias splitting method data where there is transfer learning from foreign countries to Manila.


<strong>Data split</strong>
- Train set: Singapore + Bangkok (the data-rich source domain) + the earlier period of the non-held-out Manila sensors (so the model gets some Manila fine-tuning).
- Val set: a mid time-slice (of the source cities + the non-held-out Manila) used for early stopping and hyperparameter tuning.  
- Test set: Manila's most-recent period (you ma + a held-out 1/3 of Manila sensors across all their time).  

In [52]:
# Sort out sentors based on country
df = df.sort_values('timestamp_utc').reset_index(drop=True)
corr_locations = df_loc[['location_key', 'country_iso']].apply(list, axis=1).tolist()
unique_locations = df['location_key'].unique()
PH_loc = []
TH_loc = []
SG_loc = []
#print(unique_locations)


for loc in unique_locations:
    entry = next((item for item in corr_locations if item[0] == loc), None)
    #display(entry)
    if entry[1] == 'PH':
        PH_loc.append(loc)
    elif entry[1] == 'TH':
        TH_loc.append(loc)
    elif entry[1] == 'SG':
        SG_loc.append(loc)

print(f"no. of sensors in PH: {len(PH_loc)}")
print(f"no. of sensors in TH: {len(TH_loc)}")
print(f"no. of sensors in SG: {len(SG_loc)}")
print()

# Create dataframe based on country
db_PH = df[df['location_key'].isin(PH_loc)]
db_TH = df[df['location_key'].isin(TH_loc)]
db_SG = df[df['location_key'].isin(SG_loc)]
print(f"no. of entries in PH: {len(db_PH)}")
print(f"no. of entries in TH: {len(db_TH)}")
print(f"no. of entries in SG: {len(db_SG)}")
print()

# Set up quartiles to filter time proportion
n = len(df)
p40 = int(n * 0.40)
p60 = int(n * 0.60)

# Set up held out and non-held-out Manila sensors
np.random.seed(42)
rand_PH_loc = np.random.permutation(PH_loc)
n_manila = len(PH_loc)
held_PH_loc = rand_PH_loc[0:int(n_manila/3)]
nonheld_PH_loc = rand_PH_loc[int(n_manila/3):n_manila]
db_heldPH = db_PH[db_PH['location_key'].isin(held_PH_loc)]
db_nonheldPH = db_PH[db_PH['location_key'].isin(nonheld_PH_loc)]

print(f"no. of entries in heldPH:    {len(db_heldPH)}")
print(f"no. of entries in nonheldPH: {len(db_nonheldPH)}")
print()

db_nonheldPH_early = db_nonheldPH[0:int(len(db_nonheldPH)*0.5)]
db_nonheldPH_later = db_nonheldPH[len(db_nonheldPH):]

db_TH_mid = db_TH.iloc[p40 : p60]
db_SG_mid = db_SG.iloc[p40 : p60]
db_TH_train = pd.concat([db_TH[0:p40], db_TH[p60:]], ignore_index=False)
db_SG_train = pd.concat([db_SG[0:p40], db_SG[p60:]], ignore_index=False)


# Train Configuration
train_df = pd.concat([db_TH_train, db_SG_train, db_nonheldPH_early], ignore_index=False)
train_df.sort_values('timestamp_utc').reset_index(drop=True)

# Val Configuration
val_df = pd.concat([db_TH_mid, db_SG_mid, db_nonheldPH_later], ignore_index=False)
val_df.sort_values('timestamp_utc').reset_index(drop=True)

# Test Configuration
test_df = db_heldPH

display_sizes(train_df, val_df, test_df)


no. of sensors in PH: 71
no. of sensors in TH: 187
no. of sensors in SG: 14

no. of entries in PH: 334252
no. of entries in TH: 1721378
no. of entries in SG: 63680

no. of entries in heldPH:    140311
no. of entries in nonheldPH: 193941

Train: 1458166 rows
  Val: 423862 rows
 Test: 140311 rows


## Exporting to folder

In [53]:
# Export configuration
folder_path = Path("./results")
train_filename =  "01_train.csv"
val_filename = "02_val.csv"
test_filename = "03_test.csv"

folder_path.mkdir(parents=True, exist_ok=True)
train_full_path = folder_path / train_filename
val_full_path = folder_path / val_filename
test_full_path = folder_path / test_filename

train_df.to_csv(train_full_path, index=False)
val_df.to_csv(val_full_path, index=False)
test_df.to_csv(test_full_path, index=False)